# IG visualizations

Sample 3 true positives and 3 true negatives directly from `tests/test_IG_output.jsonl`, then render the top-abundance tokens for each sample.


In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd() / "src"))

from token_source_attributor.visualization import display_random_tp_tn_token_charts


In [ ]:
# data inspection
'''
"# data inspection\n",
are there positive and negative gradients or only positive? (is there sign to this signal?
what is the mean, median, variance and mode of the IG gradients
Answers:
If abundance is 0, the gradient IS 0.

The species gradients are always 0. Since the model is Fixed Slot Attention, sequence length is an identical 1662 species vector in every sample, leading to the same exact deterministic species embedding for each sample.

The degree of abundance becomes the only sample signal provided by the data to the classifier. The species embeddings are provided by MLM pretraining.

*show them in k-mean with the species embedddings to prove they are learned


if there is sign does it mean that > 0 are drivers toward disease and < 0 are drivers toward health based on the log-odds target logit?
Yes.

[CLS] attention can tell you that this species abundance token is important but it cannot tell you why.

Integrated gradients can tell you its important and can tell you whether it leads to the diagnoses or away from it (signed gradients).
It can tease causation out of the model ether.

7382 samples

stats for all IG atributions
len:  12267222
mean -0.0011763345995437695
median 0.0
variance:  0.0663922222798484
range:  [-32.77887725830078, 36.831268310546875])

stats for IG atrributions with abundance 0
len:  11770398
mean 0.0
median 0.0
variance:  0.0
range:  [0.0, 0.0])

stats for IG atrributions with TP
len:  5665758
mean 0.002776577763198292
median 0.0
variance:  0.06383470980751133
range:  [-32.77887725830078, 23.896650314331055])


stats for IG atrributions with TN
len:  6601464
mean -0.004568952485895151
median 0.0
variance:  0.06856230674236866
range:  [-30.25447654724121, 36.831268310546875])




what is the mean median, variance and mode of all abundance scores?

stats for all abundance_scores
len:  12267222
mean 1.1133660905460094
median 0.0
mode 0
variance:  37.76271956695957
range:  [0, 50])

Abundance scores on TP samples:
stats for abundance_scores
len:  5665758
mean 0.9204831904221819
median 0.0
mode 0
variance:  31.42428089028839
range:  [0, 50])

Abundance scores on TN samples:
stats for abundance_scores
len:  6601464
mean 1.2789093449574216
median 0.0
mode 0
variance:  43.14339899830871
range:  [0, 50])

  
what is the mean median, mode and variance of the attention scores... should be bounded [0,1]
is attention being paid to tokens of 0 abundance? Yes, about an order of magnitude less than non zero abundance tokens.

stats for all attention_scores (Samples * vocab)
len:  12267222
mean 0.0006014432357802775
median 0.00030778421205468476
variance:  5.844569225386607e-06
range:  [7.09273517713882e-05, 0.222137451171875])

stats for attention scores on samples with 0 abundance 
len:  11770398
mean 0.00031654220391060515
median 0.00030326533305924386
variance:  1.245180729259016e-07
range:  [7.09273517713882e-05, 0.022158058360219002])

stats for attention scores on TP samples
len:  5665758
mean 0.000601446242652556
median 0.00034440589661244303
variance:  5.397691897813847e-06
range:  [8.064694702625275e-05, 0.20840618014335632])

stats for attention scores on TN samples
len:  6601464
mean 0.0006014406551086769
median 0.000268495234195143
variance:  6.22810517947479e-06
range:  [7.09273517713882e-05, 0.222137451171875])

'''
import json
from pathlib import Path

def load_samples(jsonl_path: str | Path) -> list[dict]:
    samples = []
    with Path(jsonl_path).open(encoding="utf-8") as jsonl_file:
        for line in jsonl_file:
            line = line.strip()
            if not line:
                continue

            record = json.loads(line)
            if record.get("record_type") == "batch":
                samples.extend(record.get("samples", []))
            else:
                samples.append(record)

    return samples

samples = load_samples("tests/test_IG_output.jsonl")
len(samples), samples[0].keys()


(7382, dict_keys(['record_type', 'dataset_path', 'checkpoint_path']))

In [87]:
'''
{'sample_number': 0,
 'sample_id': 'MV_FEI2_t1Q14',
 'study_id': '2021-03-31.AsnicarF_2017.relative_abundance',
 'disease': 'healthy',
 'label': 0,
 'pred': 0,
 'prediction_type': 'true_negative',
 'prob_healthy': 0.9992167949676514,
 'prob_ibd': 0.0007832193514332175,
 'confidence': 0.9992167949676514,
 'entropy': 0.006384559441357851,
 'tokens': [{'token_index': 0,
   'species_id': 0,
   'abundance_bin': 0,
   'ig_species': 0.0,
   'ig_abundance': 0.0,
   'ig_species_plus_abundance': 0.0,
   'cls_score': 0.0003602262295316905},
  {'token_index': 1,
   'species_id': 1,
   'abundance_bin': 0,
   'ig_species': 0.0,
   'ig_abundance': 0.0,
   'ig_species_plus_abundance': 0.0,
   'cls_score': 0.0004316550912335515},
'''
import numpy as np
meta, study_samples = samples[0],samples[1:]

sample_abundances = []
sample_attentions = []
sample_ig = []

attention_on_zero_abundances = []
for sample in study_samples:
    attention_scores = []
    sample_ig_scores = []
    if sample['prediction_type'] == 'true_negative':
        for token in sample['tokens']:
            assert token['ig_abundance'] == token['ig_species_plus_abundance']
            # if token['abundance_bin'] == 0:
            sample_ig.append(token['ig_abundance'])
            sample_attentions.append(token["cls_score"])
            sample_abundances.append(token['abundance_bin'])
            
    
    # sample_ig.append(sample_ig_scores)
    # sample_attentions.append(attention_scores)
        
abundance_scores = np.array(sample_abundances)
sample_attentions_np = np.array(sample_attentions)
sample_ig_np = np.array(sample_ig)
print('abundance scores', abundance_scores.shape[0])
print('sample attentions', sample_attentions_np.shape[0], 'len sample scores, sample_attentions_np.shape[1]')
print('sample igs ', sample_ig_np.shape[0], 'len sample scores , sample_ig_np.shape[1]')


            


abundance scores 6601464
sample attentions 6601464 len sample scores, sample_attentions_np.shape[1]
sample igs  6601464 len sample scores , sample_ig_np.shape[1]


In [88]:
def get_stats(np_array, name):
    print(f'stats for {name}')
    print('len: ', np.size(np_array))
    print(f'mean', np_array.mean())
    print(f'median', np.median(np_array))
    try:
        print(f'mode', np.bincount(np_array).argmax())
    except Exception:
        print('could not compute bincount')
    print(f'variance: ', np.var(np_array))
    print('range: ', f'[{np.min(np_array)}, {np.max(np_array)}])')
    
get_stats(abundance_scores, 'abundance_scores')
get_stats(sample_attentions_np, 'attention_scores')
get_stats(sample_ig_np, 'sample ig scores')

stats for abundance_scores
len:  6601464
mean 1.2789093449574216
median 0.0
mode 0
variance:  43.14339899830871
range:  [0, 50])
stats for attention_scores
len:  6601464
mean 0.0006014406551086769
median 0.000268495234195143
could not compute bincount
variance:  6.22810517947479e-06
range:  [7.09273517713882e-05, 0.222137451171875])
stats for sample ig scores
len:  6601464
mean -0.004568952485895151
median 0.0
could not compute bincount
variance:  0.06856230674236866
range:  [-30.25447654724121, 36.831268310546875])


stats for attention_scores
len:  7381
mean 0.0006014432357802775
median 0.00030778421205468476
variance:  5.844569225386607e-06
range:  [7.09273517713882e-05, 0.222137451171875])


In [56]:
print('range of attn on zero_abundances')
get_stats(attn_zero, 'attention paid to zero abundance samples')

range of attn on zero_abundances
stats for attention paid to zero abundance samples
len:  11770398
mean 0.00031654220391060515
median 0.00030326533305924386
variance:  1.245180729259016e-07
range:  [7.09273517713882e-05, 0.022158058360219002])


In [ ]:
display_random_tp_tn_token_charts(
    jsonl_path="tests/test_IG_output.jsonl",
    species_vocab_path="src/token_source_attributor/data/species_vocab.txt",
    samples_per_class=3,
    top_k=10,
    seed=7,
)

# maybe give more importance to visualizing items with lower abundance but higher IG?
# are there IG given to 0 abundance?



{'true_positive': [{'batch_index': 196,
   'sample_number': 6300,
   'sample_id': 'HSMA33LH',
   'study_id': '2021-10-14.HMP_2019_ibdmdb.relative_abundance',
   'disease': 'IBD',
   'label': 1,
   'pred': 1,
   'prediction_type': 'true_positive',
   'prob_healthy': 0.0008286806405521929,
   'prob_ibd': 0.9991713762283325,
   'confidence': 0.9991713762283325,
   'entropy': 0.006708329543471336,
   'tokens': [{'token_index': 0,
     'species_id': 0,
     'abundance_bin': 0,
     'ig_species': 0.0,
     'ig_abundance': 0.0,
     'ig_species_plus_abundance': 0.0,
     'cls_score': 0.0003335267538204789},
    {'token_index': 1,
     'species_id': 1,
     'abundance_bin': 0,
     'ig_species': 0.0,
     'ig_abundance': 0.0,
     'ig_species_plus_abundance': 0.0,
     'cls_score': 0.0004177129303570837},
    {'token_index': 2,
     'species_id': 2,
     'abundance_bin': 0,
     'ig_species': 0.0,
     'ig_abundance': 0.0,
     'ig_species_plus_abundance': 0.0,
     'cls_score': 0.000373689166